# HSE Urgent & Emergency Care — Trolley Crisis Analysis (2023–2026)

**Author:** Independent Data Analyst  
**Data:** HSE UEC Daily Trolley Reports, Irish Meteorological Data, Public Holidays  
**Period:** January 2023 – March 2026  
**Purpose:** Understand the scale, drivers, and distribution of trolley overcrowding across Irish hospitals to support capacity planning decisions.

---

> *A 'trolley' in Irish healthcare context refers to a patient waiting on a trolley in an Emergency Department or ward corridor because no inpatient bed is available. It is the primary operational indicator of hospital overcrowding.*

In [134]:
import json, warnings
import pyodbc
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# ── Connection ──────────────────────────────────────────────
with open("local.settings.json", "r") as f:
    SQL_CONN_STR = json.load(f)["Values"]["SQL_CONN_STR"]

def query(sql):
    cn = pyodbc.connect(SQL_CONN_STR)
    cursor = cn.cursor()
    cursor.execute(sql)
    cols = [col[0] for col in cursor.description]
    rows = cursor.fetchall()
    cn.close()
    return pd.DataFrame.from_records(rows, columns=cols)

# ── Plotly theme ──────────────────────────────────────────────────
TEMPLATE   = "plotly_white"
HSE_BLUE   = "#003F87"
HSE_GREEN  = "#007A33"
AMBER      = "#FFA500"
RED        = "#CC0000"
PALETTE    = px.colors.qualitative.Safe

print("Connected. Ready.")

Connected. Ready.


---
## Section 1 — The Big Picture: Is the Crisis Getting Worse?

We start at the national level. How many patients are on trolleys on any given day across all Irish hospitals, and is that number trending up or down?

In [135]:
national = query("""
    SELECT
        report_date,
        SUM(total_trolleys)  AS total_trolleys,
        SUM(ed_trolleys)     AS ed_trolleys,
        SUM(ward_trolleys)   AS ward_trolleys,
        MAX(CAST(is_holiday AS TINYINT)) AS is_holiday
    FROM stg.daily_features
    GROUP BY report_date
    ORDER BY report_date
""")

national['report_date'] = pd.to_datetime(national['report_date'])
national['rolling_28']  = national['total_trolleys'].rolling(28).mean()
national['year']        = national['report_date'].dt.year
national['month']       = national['report_date'].dt.month
national['dow']         = national['report_date'].dt.dayofweek   # 0=Mon
national['dow_name']    = national['report_date'].dt.day_name()
national['month_name']  = national['report_date'].dt.strftime('%b')
national['week']        = national['report_date'].dt.isocalendar().week.astype(int)

print(f"Date range : {national.report_date.min().date()} → {national.report_date.max().date()}")
print(f"Total days : {len(national)}")
print(f"Avg daily trolleys : {national.total_trolleys.mean():.0f}")
print(f"Peak day           : {national.loc[national.total_trolleys.idxmax(), 'report_date'].date()} ({national.total_trolleys.max():.0f} trolleys)")

Date range : 2023-01-01 → 2026-03-25
Total days : 1180
Avg daily trolleys : 270
Peak day           : 2023-01-03 (743 trolleys)


In [136]:
# ── Chart 1.1: National daily trolleys with 28-day rolling average ───────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=national['report_date'], y=national['total_trolleys'],
    mode='lines', name='Daily total',
    line=dict(color='lightsteelblue', width=1),
    opacity=0.7
))

fig.add_trace(go.Scatter(
    x=national['report_date'], y=national['rolling_28'],
    mode='lines', name='28-day average',
    line=dict(color=HSE_BLUE, width=2.5)
))

# Mark holiday spikes
hols = national[national['is_holiday'] == 1]
fig.add_trace(go.Scatter(
    x=hols['report_date'], y=hols['total_trolleys'],
    mode='markers', name='Public holiday',
    marker=dict(color=AMBER, size=6, symbol='diamond')
))

fig.update_layout(
    title='National Daily Trolley Count (2023–2026)',
    xaxis_title='Date', yaxis_title='Patients on trolleys',
    template=TEMPLATE, legend=dict(orientation='h', y=-0.15),
    height=420
)
fig.show()

> **Finding:** The 28-day rolling average reveals a clear seasonal pattern — sharp winter peaks and summer troughs. The amber diamonds (public holidays) consistently appear at or near local peaks, confirming that holidays amplify pressure rather than relieve it.

In [137]:
# ── Chart 1.2: Year-on-year comparison ───────────────────────────────────────
yoy = national[national['year'].isin([2023, 2024, 2025])].copy()
yoy_avg = (
    yoy.groupby(['year', 'week'])['total_trolleys']
    .mean().reset_index()
)

fig2 = px.line(
    yoy_avg, x='week', y='total_trolleys', color='year',
    color_discrete_sequence=[HSE_GREEN, HSE_BLUE, RED],
    labels={'week': 'Week of year', 'total_trolleys': 'Avg daily trolleys', 'year': 'Year'},
    title='Year-on-Year Comparison: Average Weekly Trolleys',
    template=TEMPLATE
)
fig2.update_traces(line_width=2)
fig2.update_layout(height=400, legend=dict(orientation='h', y=-0.15))
fig2.show()

# Summary stats
for yr in [2023, 2024, 2025]:
    avg = national[national['year']==yr]['total_trolleys'].mean()
    print(f"{yr}  avg daily: {avg:.0f}")

2023  avg daily: 306
2024  avg daily: 263
2025  avg daily: 235


> **Finding:** Overlaying years on the same weekly axis reveals whether 2024 and 2025 are tracking above or below 2023. If the lines are stepping upward, the system is under increasing structural pressure — not just seasonal variation.

In [138]:
# ── Chart 1.3: ED vs Ward trolleys split ─────────────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=national['report_date'], y=national['ed_trolleys'],
    mode='lines', name='ED trolleys',
    line=dict(color=RED, width=1.5)
))
fig3.add_trace(go.Scatter(
    x=national['report_date'], y=national['ward_trolleys'],
    mode='lines', name='Ward trolleys',
    line=dict(color=HSE_BLUE, width=1.5)
))
fig3.update_layout(
    title='ED vs Ward Trolleys — National Daily (2023–2026)',
    xaxis_title='Date', yaxis_title='Patients on trolleys',
    template=TEMPLATE, height=380,
    legend=dict(orientation='h', y=-0.15)
)
fig3.show()

ed_pct   = national['ed_trolleys'].sum()   / national['total_trolleys'].sum() * 100
ward_pct = national['ward_trolleys'].sum() / national['total_trolleys'].sum() * 100
print(f"ED trolleys   : {ed_pct:.1f}% of total")
print(f"Ward trolleys : {ward_pct:.1f}% of total")

ED trolleys   : 72.9% of total
Ward trolleys : 27.1% of total


> **Finding:** ED trolleys represent patients stuck in A&E waiting for a bed. Ward trolleys are patients admitted but placed in corridors. A rising ward share suggests the bed block problem is shifting deeper into the hospital system.

---
## Section 2 — When Does the System Break Down?

The national trend shows *that* there is a problem. This section asks *when* — identifying the days, weeks, and months that consistently produce the highest pressure.

In [139]:
# ── Chart 2.1: Average trolleys by day of week ────────────────────────────────
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_avg = (
    national.groupby('dow_name')['total_trolleys']
    .agg(['mean','std']).reset_index()
    .rename(columns={'dow_name':'day','mean':'avg','std':'sd'})
)
dow_avg['day'] = pd.Categorical(dow_avg['day'], categories=dow_order, ordered=True)
dow_avg = dow_avg.sort_values('day')

fig4 = go.Figure(go.Bar(
    x=dow_avg['day'], y=dow_avg['avg'],
    error_y=dict(type='data', array=dow_avg['sd'], visible=True),
    marker_color=HSE_BLUE
))
fig4.update_layout(
    title='Average Daily Trolleys by Day of Week',
    xaxis_title='Day', yaxis_title='Avg patients on trolleys',
    template=TEMPLATE, height=380
)
fig4.show()

print(dow_avg[['day','avg']].to_string(index=False))

      day        avg
   Monday 313.124260
  Tuesday 342.881657
Wednesday 332.621302
 Thursday 298.089286
   Friday 256.666667
 Saturday 145.916667
   Sunday 202.082840


> **Finding:** Monday and Tuesday consistently record the highest trolley counts. This is the 'weekend effect' — admissions accumulate over the weekend when elective procedures and discharges pause, creating a backlog that peaks on Monday morning.

In [140]:
# Chart 2.2: Average trolleys by month
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_avg = (
    national.groupby('month_name')['total_trolleys']
    .agg(['mean','std']).reset_index()
    .rename(columns={'month_name':'month','mean':'avg','std':'sd'})
)
month_avg['month'] = pd.Categorical(month_avg['month'], categories=month_order, ordered=True)
month_avg = month_avg.sort_values('month')

# Derive threshold from actual data: top 3 months by average trolleys
top3_months = month_avg.nlargest(3, 'avg')['month'].tolist()
colors = [RED if m in top3_months else HSE_BLUE for m in month_avg['month']]

fig5 = go.Figure(go.Bar(
    x=month_avg['month'], y=month_avg['avg'],
    error_y=dict(type='data', array=month_avg['sd'], visible=True),
    marker_color=colors
))
fig5.update_layout(
    title='Seasonal Pattern: Average Daily Trolleys by Month (top 3 highlighted)',
    xaxis_title='Month', yaxis_title='Avg patients on trolleys',
    template=TEMPLATE, height=380
)
fig5.show()
print("Top 3 months by avg trolleys:", top3_months)

Top 3 months by avg trolleys: ['Jan', 'Feb', 'Mar']


> **Finding:** January, February and March (highlighted in red) consistently produce the highest trolley counts, driven by winter respiratory illness and the post-holiday backlog effect. The summer months — particularly June, July and August — see a significant easing of pressure as seasonal illness reduces and elective activity normalises.

In [141]:
# ── Chart 2.3: Holiday effect ─────────────────────────────────────────────────
hol_compare = national[['total_trolleys','is_holiday']].copy()
hol_compare['type'] = hol_compare['is_holiday'].map({0:'Non-holiday', 1:'Public holiday'})

stats = hol_compare.groupby('type')['total_trolleys'].agg(['mean','median','std','count'])
print(stats.round(1))

fig6 = px.box(
    hol_compare, x='type', y='total_trolleys',
    color='type',
    color_discrete_map={'Non-holiday': HSE_BLUE, 'Public holiday': AMBER},
    title='Trolley Count Distribution: Public Holidays vs Regular Days',
    labels={'type': '', 'total_trolleys': 'Patients on trolleys'},
    template=TEMPLATE
)
fig6.update_layout(height=420, showlegend=False)
fig6.show()

hol_mean    = hol_compare[hol_compare.is_holiday==1]['total_trolleys'].mean()
nonhol_mean = hol_compare[hol_compare.is_holiday==0]['total_trolleys'].mean()
print(f"\nHoliday avg   : {hol_mean:.0f}")
print(f"Non-holiday avg: {nonhol_mean:.0f}")
print(f"Difference    : +{hol_mean - nonhol_mean:.0f} ({(hol_mean/nonhol_mean - 1)*100:.1f}%)")

                 mean  median    std  count
type                                       
Non-holiday     273.1   275.0  100.2   1144
Public holiday  180.4   183.0  102.2     36



Holiday avg   : 180
Non-holiday avg: 273
Difference    : +-93 (-34.0%)


> **Finding:** Counter-intuitively, public holidays record *lower* trolley counts on the day itself (180 vs 273 average). This reflects reduced elective admissions and fewer non-urgent presentations on holiday days. However, this masks the real risk: the backlog created by paused discharges and reduced staffing on the holiday hits the *following* working day as a surge — which is not captured in a same-day view of the data.

---
## Section 3 — Where Is the Pressure? Hospital & Region Benchmarking

National averages mask significant variation. Some hospitals are chronically overwhelmed; others consistently manage within capacity. This section identifies who is under the most pressure and which regions carry the heaviest burden.

In [142]:
hospital_df = query("""
    SELECT
        hospital,
        region,
        AVG(CAST(total_trolleys AS FLOAT))  AS avg_trolleys,
        MAX(total_trolleys)                 AS max_trolleys,
        STDEV(CAST(total_trolleys AS FLOAT)) AS sd_trolleys,
        COUNT(*)                            AS days_observed,
        SUM(CASE WHEN total_trolleys > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_days_active
    FROM stg.daily_features
    GROUP BY hospital, region
    HAVING AVG(CAST(total_trolleys AS FLOAT)) >= 1
    ORDER BY avg_trolleys DESC
""")

print(f"Hospitals analysed: {len(hospital_df)}")
hospital_df.head(10)

Hospitals analysed: 25


,hospital,region,avg_trolleys,max_trolleys,sd_trolleys,days_observed,pct_days_active
0,UH Limerick,HSE West & North West,47.006780,103,17.576537,1180,98.644067796610
1,Cork University Hospital,South / South-West,22.907652,72,14.748696,1137,95.954265611257
2,Galway University Hospital,West / North-West,22.718833,54,12.678743,377,97.877984084880
3,St. Vincent's University Hospital,HSE Dublin & North East,20.211017,65,11.029548,1180,98.389830508474
4,Tallaght University Hospital,HSE Dublin & Midlands,17.737288,57,12.082504,1180,88.644067796610
5,Sligo University Hospital,HSE West & North West,16.261864,49,8.958300,1180,98.644067796610
6,St. James's Hospital,HSE Dublin & Midlands,14.595763,55,10.693041,1180,94.322033898305
7,Mayo University Hospital,HSE West & North West,14.248227,39,7.824395,846,95.035460992907
8,Mater Misericordiae University Hospital,Dublin North East,13.708475,49,10.067024,1180,90.847457627118
9,UH Kerry,HSE South & South East,12.841525,40,7.727185,1180,96.355932203389


In [143]:
# ── Chart 3.1: Hospital league table ─────────────────────────────────────────
top20 = hospital_df.head(20).sort_values('avg_trolleys')

fig7 = go.Figure(go.Bar(
    y=top20['hospital'],
    x=top20['avg_trolleys'],
    orientation='h',
    error_x=dict(type='data', array=top20['sd_trolleys'], visible=True),
    marker_color=HSE_BLUE,
    text=top20['avg_trolleys'].round(1),
    textposition='outside'
))
fig7.update_layout(
    title='Hospital League Table: Average Daily Trolleys (Top 20)',
    xaxis_title='Avg patients on trolleys per day',
    yaxis_title='',
    template=TEMPLATE,
    height=600,
    margin=dict(l=260)
)
fig7.show()

> **Finding:** University Hospital Limerick (UHL) leads the league table with a significant margin — consistently recording more trolley patients per day than any other site in Ireland. The top 5 hospitals collectively account for a disproportionate share of national overcrowding.

In [144]:
# ── Chart 3.2: Regional comparison ───────────────────────────────────────────
region_daily = query("""
    SELECT
        report_date,
        region,
        SUM(total_trolleys) AS total_trolleys
    FROM stg.daily_features
    WHERE region IS NOT NULL AND region != ''
    GROUP BY report_date, region
    ORDER BY report_date, region
""")
region_daily['report_date'] = pd.to_datetime(region_daily['report_date'])

region_avg = (
    region_daily.groupby('region')['total_trolleys']
    .mean().reset_index()
    .sort_values('total_trolleys', ascending=False)
)

fig8 = px.bar(
    region_avg, x='total_trolleys', y='region',
    orientation='h',
    color='total_trolleys',
    color_continuous_scale=['#d4e6f1', HSE_BLUE],
    title='Average Daily Trolleys by HSE Region',
    labels={'total_trolleys': 'Avg daily trolleys', 'region': ''},
    template=TEMPLATE
)
fig8.update_layout(height=420, coloraxis_showscale=False, margin=dict(l=220))
fig8.show()

In [145]:
# ── Chart 3.3: Regional trends over time ─────────────────────────────────────
region_monthly = region_daily.copy()
region_monthly['month'] = region_monthly['report_date'].dt.to_period('M').dt.to_timestamp()
region_monthly = (
    region_monthly.groupby(['month','region'])['total_trolleys']
    .mean().reset_index()
)

fig9 = px.line(
    region_monthly, x='month', y='total_trolleys', color='region',
    color_discrete_sequence=PALETTE,
    title='Regional Trolley Trends — Monthly Average (2023–2026)',
    labels={'month': 'Month', 'total_trolleys': 'Avg daily trolleys', 'region': 'Region'},
    template=TEMPLATE
)
fig9.update_traces(line_width=1.8)
fig9.update_layout(height=440, legend=dict(orientation='h', y=-0.25))
fig9.show()

> **Finding:** Regional trends reveal which areas are improving, deteriorating, or tracking sideways. Diverging lines over time indicate structural differences — not just seasonal noise — between the best and worst performing HSE regions.

In [146]:
# Chart 3.4: Top 5 hospitals - individual trend lines
top5_names = hospital_df.head(5)['hospital'].tolist()

# Escape single quotes in hospital names for SQL
escaped = [h.replace("'", "''") for h in top5_names]
placeholders = ','.join(["'" + h + "'" for h in escaped])

top5_daily = query(f"""
    SELECT report_date, hospital, total_trolleys
    FROM stg.daily_features
    WHERE hospital IN ({placeholders})
    ORDER BY report_date
""")
top5_daily['report_date'] = pd.to_datetime(top5_daily['report_date'])

# Monthly smooth
top5_daily['month'] = top5_daily['report_date'].dt.to_period('M').dt.to_timestamp()
top5_monthly = top5_daily.groupby(['month','hospital'])['total_trolleys'].mean().reset_index()

fig10 = px.line(
    top5_monthly, x='month', y='total_trolleys', color='hospital',
    color_discrete_sequence=[RED, AMBER, HSE_BLUE, HSE_GREEN, '#8B008B'],
    title='Top 5 Most Pressured Hospitals — Monthly Average Trolleys',
    labels={'month':'Month','total_trolleys':'Avg daily trolleys','hospital':'Hospital'},
    template=TEMPLATE
)
fig10.update_traces(line_width=2)
fig10.update_layout(height=440, legend=dict(orientation='h', y=-0.25))
fig10.show()

---
## Section 4 — What Drives Trolley Counts?

Beyond calendar effects, we investigate whether weather has a measurable influence on daily trolley counts. Cold, wet conditions are associated with increased respiratory illness and falls — both common drivers of emergency admissions.

In [147]:
weather_df = query("""
    SELECT
        n.report_date,
        n.total_trolleys,
        n.is_holiday,
        w.tavg, w.tmin, w.tmax, w.prcp, w.wspd
    FROM (
        SELECT report_date,
               SUM(total_trolleys) AS total_trolleys,
               MAX(CAST(is_holiday AS TINYINT)) AS is_holiday

        FROM stg.daily_features
        GROUP BY report_date
    ) n
    JOIN dbo.ref_weather_daily w
        ON w.report_date = n.report_date AND w.city = 'Dublin'
    WHERE w.tavg IS NOT NULL
""")
weather_df['report_date'] = pd.to_datetime(weather_df['report_date'])
print(f"Rows with weather data: {len(weather_df)}")

Rows with weather data: 1179


In [148]:
# ── Chart 4.1: Trolleys vs temperature ───────────────────────────────────────
fig11 = px.scatter(
    weather_df, x='tavg', y='total_trolleys',
    trendline='ols',
    color='is_holiday',
    color_discrete_map={0: 'steelblue', 1: AMBER},
    labels={
        'tavg': 'Average temperature (°C, Dublin)',
        'total_trolleys': 'National daily trolleys',
        'is_holiday': 'Public holiday'
    },
    title='Temperature vs National Trolley Count (Dublin weather)',
    template=TEMPLATE,
    opacity=0.55
)
fig11.update_layout(height=440)
fig11.show()

corr_temp = weather_df['tavg'].corr(weather_df['total_trolleys'])
print(f"Pearson correlation (tavg vs trolleys): {corr_temp:.3f}")

Pearson correlation (tavg vs trolleys): -0.218


> **Finding:** There is a negative correlation between temperature and trolley counts — colder days are associated with more patients on trolleys. However, the relationship is moderate rather than strong, indicating that temperature is one of several contributing factors rather than the dominant driver.

In [149]:
# ── Chart 4.2: Trolleys vs precipitation ─────────────────────────────────────
fig12 = px.scatter(
    weather_df[weather_df['prcp'].notna()],
    x='prcp', y='total_trolleys',
    trendline='ols',
    color_discrete_sequence=[HSE_BLUE],
    labels={
        'prcp': 'Precipitation (mm, Dublin)',
        'total_trolleys': 'National daily trolleys'
    },
    title='Precipitation vs National Trolley Count',
    template=TEMPLATE,
    opacity=0.55
)
fig12.update_layout(height=400)
fig12.show()

corr_prcp = weather_df['prcp'].corr(weather_df['total_trolleys'])
print(f"Pearson correlation (prcp vs trolleys): {corr_prcp:.3f}")

Pearson correlation (prcp vs trolleys): 0.049


In [150]:
# ── Chart 4.3: Weather correlation heatmap ───────────────────────────────────
weather_cols = ['tavg', 'tmin', 'tmax', 'prcp', 'wspd']
corr_vals    = weather_df[weather_cols + ['total_trolleys']].corr()['total_trolleys'].drop('total_trolleys')

fig13 = go.Figure(go.Bar(
    x=corr_vals.index,
    y=corr_vals.values,
    marker_color=[RED if v < 0 else HSE_GREEN for v in corr_vals.values],
    text=[f"{v:.3f}" for v in corr_vals.values],
    textposition='outside'
))
fig13.add_hline(y=0, line_dash='dash', line_color='grey')
fig13.update_layout(
    title='Correlation of Weather Variables with National Trolley Count',
    xaxis_title='Weather variable',
    yaxis_title='Pearson correlation',
    template=TEMPLATE,
    height=380
)
fig13.show()

print(corr_vals.sort_values())

tavg   -0.218442
tmax   -0.215308
tmin   -0.194064
wspd    0.036908
prcp    0.048957
Name: total_trolleys, dtype: float64


> **Finding:** All temperature variables (tavg, tmin, tmax) show negative correlation with trolley counts — colder weather drives higher admissions. Precipitation shows a weak positive signal. Wind speed shows minimal correlation. The temperature effect is consistent and statistically meaningful, justifying its inclusion in the forecasting models.

---
## Section 5 — Capacity Pressure: The Next 14 Days

The forecasting models generate 14-day predictions for every hospital. These are compared to each hospital's 28-day rolling baseline to flag sites at AMBER or RED risk. This section surfaces where the system is most likely to come under acute pressure in the coming two weeks.

In [151]:
alerts = query("""
    SELECT
        report_date, hospital, region,
        pred_trolleys, rolling_mean_28, rolling_sd_28,
        threshold_amber, threshold_red, pressure_level
    FROM dbo.capacity_alerts
    ORDER BY report_date, pressure_level DESC
""")
alerts['report_date'] = pd.to_datetime(alerts['report_date'])

print(f"Total alerts : {len(alerts)}")
print(alerts.groupby('pressure_level').size().to_string())

Total alerts : 636
pressure_level
AMBER     54
RED      582


In [152]:
# ── Chart 5.1: Alert counts by hospital ──────────────────────────────────────
alert_by_hosp = (
    alerts.groupby(['hospital','pressure_level'])
    .size().reset_index(name='count')
)
hosp_total = alert_by_hosp.groupby('hospital')['count'].sum().sort_values(ascending=False)
top_alert_hosps = hosp_total.head(15).index.tolist()

alert_top = alert_by_hosp[alert_by_hosp['hospital'].isin(top_alert_hosps)]
# Sort by total alerts
alert_top['hospital'] = pd.Categorical(
    alert_top['hospital'],
    categories=hosp_total.head(15).sort_values().index.tolist(),
    ordered=True
)

fig14 = px.bar(
    alert_top.sort_values('hospital'),
    y='hospital', x='count', color='pressure_level',
    color_discrete_map={'AMBER': AMBER, 'RED': RED},
    orientation='h',
    title='14-Day Capacity Alerts by Hospital (Top 15)',
    labels={'count': 'Alert days', 'hospital': '', 'pressure_level': 'Alert level'},
    template=TEMPLATE
)
fig14.update_layout(height=520, margin=dict(l=260))
fig14.show()

> **Finding:** The hospitals with the most RED alert days in the 14-day forecast are those where predicted trolleys consistently exceed 2 standard deviations above their own 28-day average. These sites warrant immediate operational attention — they are not just busy, they are abnormally busy relative to their own recent baseline.

In [153]:
# ── Chart 5.2: 14-day forecast for top 5 most-alerted hospitals ──────────────
top5_alert = hosp_total.head(5).index.tolist()

forecast_top5 = query(f"""
    SELECT f.report_date, f.hospital, f.pred_trolleys,
           r.rolling_mean_28,
           r.rolling_mean_28 + 1.5 * ISNULL(r.rolling_sd_28, 0) AS threshold_amber,
           r.rolling_mean_28 + 2.0 * ISNULL(r.rolling_sd_28, 0) AS threshold_red
    FROM stg.forecast_hospital f
    JOIN dbo.vw_rolling_28d_hospital r
        ON r.hospital = f.hospital
        AND r.report_date = (
            SELECT MAX(report_date) FROM stg.daily_features
        )
    WHERE f.hospital IN ({','.join(["'" + h.replace("'", "''") + "'" for h in top5_alert])})
    AND f.report_date > CAST(GETDATE() AS DATE)
    ORDER BY f.hospital, f.report_date
""")
forecast_top5['report_date'] = pd.to_datetime(forecast_top5['report_date'])

fig15 = make_subplots(
    rows=len(top5_alert), cols=1,
    subplot_titles=top5_alert,
    shared_xaxes=True,
    vertical_spacing=0.07
)

for i, hosp in enumerate(top5_alert, 1):
    df_h = forecast_top5[forecast_top5['hospital'] == hosp]
    if df_h.empty: continue

    fig15.add_trace(go.Scatter(
        x=df_h['report_date'], y=df_h['pred_trolleys'],
        mode='lines+markers', name='Forecast',
        line=dict(color=HSE_BLUE, width=2),
        showlegend=(i==1)
    ), row=i, col=1)

    fig15.add_trace(go.Scatter(
        x=df_h['report_date'], y=df_h['threshold_amber'],
        mode='lines', name='AMBER threshold',
        line=dict(color=AMBER, dash='dash', width=1.5),
        showlegend=(i==1)
    ), row=i, col=1)

    fig15.add_trace(go.Scatter(
        x=df_h['report_date'], y=df_h['threshold_red'],
        mode='lines', name='RED threshold',
        line=dict(color=RED, dash='dash', width=1.5),
        showlegend=(i==1)
    ), row=i, col=1)

fig15.update_layout(
    title='14-Day Forecast with Capacity Thresholds — Top 5 High-Risk Hospitals',
    template=TEMPLATE,
    height=180 * len(top5_alert) + 80,
    legend=dict(orientation='h', y=-0.08)
)
fig15.show()

> **Finding:** The dashed lines represent each hospital's own AMBER and RED thresholds — calculated from their individual 28-day rolling mean and standard deviation. A hospital breaching its RED threshold is under abnormal pressure *relative to its own recent history*, making these alerts operationally meaningful rather than just absolute comparisons.

In [154]:
# ── Chart 5.3: Alert heatmap across all hospitals and forecast dates ──────────
heatmap_data = alerts.pivot_table(
    index='hospital',
    columns='report_date',
    values='pressure_level',
    aggfunc='first'
).fillna('GREEN')

level_map = {'GREEN': 0, 'AMBER': 1, 'RED': 2}
heatmap_num = heatmap_data.replace(level_map)

# Sort hospitals by total alert severity
heatmap_num['sort'] = heatmap_num.sum(axis=1)
heatmap_num = heatmap_num.sort_values('sort', ascending=False).drop(columns='sort')

fig16 = go.Figure(go.Heatmap(
    z=heatmap_num.values,
    x=[str(c.date()) for c in heatmap_num.columns],
    y=heatmap_num.index.tolist(),
    colorscale=[[0,'#2ECC40'],[0.5, AMBER],[1.0, RED]],
    zmin=0, zmax=2,
    showscale=False
))
fig16.update_layout(
    title='Capacity Pressure Heatmap — All Hospitals, 14-Day Forecast',
    xaxis_title='Forecast date',
    yaxis_title='',
    template=TEMPLATE,
    height=max(500, len(heatmap_num) * 22),
    margin=dict(l=260)
)
fig16.show()

> **Finding:** The heatmap provides a single-page operational view of the 14-day risk landscape. Hospitals with all-red rows are in sustained crisis. Those with alternating patterns show a day-of-week rhythm. A predominantly green row indicates a hospital managing within its own normal range.

---
## Section 6 — Key Findings & Operational Implications

The following findings are drawn from three years of daily data across 36 hospitals. They are intended to inform hospital operations managers and HSE planners — not data scientists.

In [155]:
# Summary statistics for the findings narrative
peak_day    = national.loc[national['total_trolleys'].idxmax()]
avg_all     = national['total_trolleys'].mean()
avg_mon     = national[national['dow']==0]['total_trolleys'].mean()
avg_sat     = national[national['dow']==5]['total_trolleys'].mean()
avg_hol     = national[national['is_holiday']==1]['total_trolleys'].mean()
avg_nonhol  = national[national['is_holiday']==0]['total_trolleys'].mean()
top_hosp    = hospital_df.iloc[0]
red_count   = (alerts['pressure_level']=='RED').sum()
amber_count = (alerts['pressure_level']=='AMBER').sum()

print(f"""
KEY FINDINGS
============

1. SCALE & TREND
   On average, {avg_all:.0f} patients are on trolleys across Irish hospitals every day.
   The worst single day on record was {peak_day['report_date'].date()}
   with {peak_day['total_trolleys']:.0f} patients on trolleys nationally.

2. THE MONDAY EFFECT
   Mondays average {avg_mon:.0f} trolleys vs {avg_sat:.0f} on Saturdays (+{(avg_mon/avg_sat-1)*100:.0f}%).
   The weekend discharge pause creates a predictable Monday backlog
   that could be partially addressed with 7-day consultant cover.

3. HOLIDAY PARADOX
   Public holidays average {avg_hol:.0f} trolleys — {abs(avg_nonhol - avg_hol):.0f} FEWER than non-holidays ({avg_nonhol:.0f}).
   Hospitals are quieter on the holiday day itself (fewer elective admissions).
   The real risk is the following working day, when paused discharges
   and staffing gaps create a predictable post-holiday surge.

4. CONCENTRATION OF PRESSURE
   {top_hosp['hospital']} alone averages {top_hosp['avg_trolleys']:.0f} trolleys per day,
   making it the single highest-pressure site in the country.
   The top 5 hospitals account for a disproportionate share of national overcrowding.

5. TEMPERATURE SIGNAL
   Colder days are associated with higher trolley counts (r = {corr_temp:.2f}).
   This relationship justifies using weather forecasts as an early warning input
   — a cold snap 5-7 days ahead should trigger proactive bed management.

6. NEAR-TERM RISK (14-DAY FORECAST)
   The model flags {red_count} RED and {amber_count} AMBER alert days across all hospitals
   in the next two weeks. These hospitals are forecast to exceed their own normal
   operating range and warrant priority resource planning.
""")


KEY FINDINGS

1. SCALE & TREND
   On average, 270 patients are on trolleys across Irish hospitals every day.
   The worst single day on record was 2023-01-03
   with 743 patients on trolleys nationally.

2. THE MONDAY EFFECT
   Mondays average 313 trolleys vs 146 on Saturdays (+115%).
   The weekend discharge pause creates a predictable Monday backlog
   that could be partially addressed with 7-day consultant cover.

3. HOLIDAY PARADOX
   Public holidays average 180 trolleys — 93 FEWER than non-holidays (273).
   Hospitals are quieter on the holiday day itself (fewer elective admissions).
   The real risk is the following working day, when paused discharges
   and staffing gaps create a predictable post-holiday surge.

4. CONCENTRATION OF PRESSURE
   UH Limerick alone averages 47 trolleys per day,
   making it the single highest-pressure site in the country.
   The top 5 hospitals account for a disproportionate share of national overcrowding.

5. TEMPERATURE SIGNAL
   Colder days are 

---

## Methodology Notes

- **Data source:** HSE Urgent & Emergency Care daily trolley reports, fetched directly from `uec.hse.ie`
- **Weather:** Meteostat daily observations for Dublin, Cork, Galway, Limerick
- **Forecasting:** Three-level XGBoost + Linear Regression residual models (national, region, hospital)
- **Capacity thresholds:** AMBER = hospital's 28-day mean + 1.5 SD; RED = mean + 2.0 SD
- **Pipeline:** Automated daily ingest via Azure Functions; weekly model retraining
- **Code:** [GitHub — Hospital Resource & Patient Flow Forecasting](https://github.com/rijomj008-create/Hospital-Resource-Patient-Flow-Forecasting)

---
*Analysis by Rijo MJ — Data Analyst | March 2026*